In [1]:
import pandas as pd
import os

portfolio_path = "../reports/day22_portfolio.csv"

portfolio = pd.read_csv(portfolio_path)

print(portfolio.shape)
print(portfolio.columns.tolist())

portfolio.head()

(5, 10)
['portfolio_rank', 'company_id', 'recommendation', 'allocation_percent', 'composite_score', 'quality_score_100', 'momentum_score_100', 'normalized_risk_score', 'risk_category', 'trend_classification']


,portfolio_rank,company_id,recommendation,allocation_percent,composite_score,quality_score_100,momentum_score_100,normalized_risk_score,risk_category,trend_classification
0,1,BEL,Strong Buy,32.39,73.189325,100.00,85.71,6.67,Low Risk,Strong Improvement
1,2,HAL,Buy,25.76,58.206714,79.45,42.86,13.33,Low Risk,Declining
2,3,INDIGO,Hold,16.61,37.526171,51.09,57.14,6.67,Low Risk,Improving
3,4,ABB,Hold,14.06,31.765038,43.19,71.43,13.33,Low Risk,Improving
4,5,TRENT,Watch,11.19,25.274906,34.29,100.00,6.67,Low Risk,Strong Improvement


In [2]:
price_path = "../data/raw/supporting/stock_prices.xlsx"

prices = pd.read_excel(price_path)

print(prices.shape)
print(prices.columns.tolist())

prices.head()

(5520, 9)
['id', 'company_id', 'date', 'open_price', 'high_price', 'low_price', 'close_price', 'volume', 'adjusted_close']


,id,company_id,date,open_price,high_price,low_price,close_price,volume,adjusted_close
0,1,ABB,2020-01-01,1958.61,2150.53,1835.28,1964.30,26184368,1964.30
1,2,ABB,2020-02-01,2177.03,2192.92,1931.08,2165.21,1668316,2165.21
2,3,ABB,2020-03-01,2221.26,2440.28,1995.23,2198.78,20578107,2198.78
3,4,ABB,2020-04-01,1932.02,2027.44,1740.12,1958.56,14558597,1958.56
4,5,ABB,2020-05-01,2089.00,2290.96,1878.23,2085.72,44638694,2085.72


In [3]:
for col in prices.columns:
    print(col)

id
company_id
date
open_price
high_price
low_price
close_price
volume
adjusted_close


In [4]:
price_cols = []

for col in prices.columns:
    if any(word in col.lower() for word in [
        "close",
        "price",
        "current",
        "latest",
        "cmp"
    ]):
        price_cols.append(col)

price_cols

['open_price', 'high_price', 'low_price', 'close_price', 'adjusted_close']

In [5]:
history_cols = []

for col in prices.columns:
    if any(word in col.lower() for word in [
        "open",
        "high",
        "low",
        "previous",
        "prev",
        "52"
    ]):
        history_cols.append(col)

history_cols

['open_price', 'high_price', 'low_price']

In [6]:
prices["date"] = pd.to_datetime(prices["date"])

prices = prices.sort_values(
    ["company_id", "date"]
)

prices.head()

,id,company_id,date,open_price,high_price,low_price,close_price,volume,adjusted_close
0,1,ABB,2020-01-01,1958.61,2150.53,1835.28,1964.30,26184368,1964.30
1,2,ABB,2020-02-01,2177.03,2192.92,1931.08,2165.21,1668316,2165.21
2,3,ABB,2020-03-01,2221.26,2440.28,1995.23,2198.78,20578107,2198.78
3,4,ABB,2020-04-01,1932.02,2027.44,1740.12,1958.56,14558597,1958.56
4,5,ABB,2020-05-01,2089.00,2290.96,1878.23,2085.72,44638694,2085.72


In [7]:
latest_prices = (
    prices
    .groupby("company_id")
    .last()
    .reset_index()
)

latest_prices = latest_prices[
    [
        "company_id",
        "close_price"
    ]
]

latest_prices.head()

,company_id,close_price
0,ABB,4913.07
1,ADANIENSOL,3701.81
2,ADANIENT,8686.54
3,ADANIGREEN,7223.66
4,ADANIPORTS,1343.11


In [8]:
first_prices = (
    prices
    .groupby("company_id")
    .first()
    .reset_index()
)

first_prices = first_prices[
    [
        "company_id",
        "close_price"
    ]
]

first_prices = first_prices.rename(
    columns={
        "close_price":"initial_price"
    }
)

first_prices.head()

,company_id,initial_price
0,ABB,1964.30
1,ADANIENSOL,4831.28
2,ADANIENT,3729.19
3,ADANIGREEN,3060.87
4,ADANIPORTS,883.73


In [9]:
performance = portfolio.merge(
    first_prices,
    on="company_id"
)

performance = performance.merge(
    latest_prices,
    on="company_id"
)

performance.head()

,portfolio_rank,company_id,recommendation,allocation_percent,composite_score,quality_score_100,momentum_score_100,normalized_risk_score,risk_category,trend_classification,initial_price,close_price
0,1,BEL,Strong Buy,32.39,73.189325,100.00,85.71,6.67,Low Risk,Strong Improvement,1720.80,1998.73
1,2,HAL,Buy,25.76,58.206714,79.45,42.86,13.33,Low Risk,Declining,4440.64,7687.57
2,3,INDIGO,Hold,16.61,37.526171,51.09,57.14,6.67,Low Risk,Improving,1579.29,5956.44
3,4,ABB,Hold,14.06,31.765038,43.19,71.43,13.33,Low Risk,Improving,1964.30,4913.07
4,5,TRENT,Watch,11.19,25.274906,34.29,100.00,6.67,Low Risk,Strong Improvement,598.61,835.01


In [10]:
performance["return_percent"] = (
    (
        performance["close_price"]
        -
        performance["initial_price"]
    )
    /
    performance["initial_price"]
) * 100

performance["return_percent"] = performance[
    "return_percent"
].round(2)

performance.head()

,portfolio_rank,company_id,recommendation,allocation_percent,composite_score,quality_score_100,momentum_score_100,normalized_risk_score,risk_category,trend_classification,initial_price,close_price,return_percent
0,1,BEL,Strong Buy,32.39,73.189325,100.00,85.71,6.67,Low Risk,Strong Improvement,1720.80,1998.73,16.15
1,2,HAL,Buy,25.76,58.206714,79.45,42.86,13.33,Low Risk,Declining,4440.64,7687.57,73.12
2,3,INDIGO,Hold,16.61,37.526171,51.09,57.14,6.67,Low Risk,Improving,1579.29,5956.44,277.16
3,4,ABB,Hold,14.06,31.765038,43.19,71.43,13.33,Low Risk,Improving,1964.30,4913.07,150.12
4,5,TRENT,Watch,11.19,25.274906,34.29,100.00,6.67,Low Risk,Strong Improvement,598.61,835.01,39.49


In [11]:
def performance_label(x):
    if x >= 20:
        return "Excellent"

    elif x >= 10:
        return "Good"

    elif x >= 0:
        return "Average"

    else:
        return "Poor"


performance["performance"] = performance[
    "return_percent"
].apply(performance_label)

performance.head()

,portfolio_rank,company_id,recommendation,allocation_percent,composite_score,quality_score_100,momentum_score_100,normalized_risk_score,risk_category,trend_classification,initial_price,close_price,return_percent,performance
0,1,BEL,Strong Buy,32.39,73.189325,100.00,85.71,6.67,Low Risk,Strong Improvement,1720.80,1998.73,16.15,Good
1,2,HAL,Buy,25.76,58.206714,79.45,42.86,13.33,Low Risk,Declining,4440.64,7687.57,73.12,Excellent
2,3,INDIGO,Hold,16.61,37.526171,51.09,57.14,6.67,Low Risk,Improving,1579.29,5956.44,277.16,Excellent
3,4,ABB,Hold,14.06,31.765038,43.19,71.43,13.33,Low Risk,Improving,1964.30,4913.07,150.12,Excellent
4,5,TRENT,Watch,11.19,25.274906,34.29,100.00,6.67,Low Risk,Strong Improvement,598.61,835.01,39.49,Excellent


In [12]:
summary = pd.DataFrame({

    "Metric":[
        "Companies",
        "Average Return",
        "Maximum Return",
        "Minimum Return"
    ],

    "Value":[
        len(performance),
        performance["return_percent"].mean(),
        performance["return_percent"].max(),
        performance["return_percent"].min()
    ]

})

summary

,Metric,Value
0,Companies,5.000
1,Average Return,111.208
2,Maximum Return,277.160
3,Minimum Return,16.150


In [13]:
performance.to_csv(
    "../reports/day23_portfolio_performance.csv",
    index=False
)

summary.to_csv(
    "../reports/day23_portfolio_summary.csv",
    index=False
)

print("Reports Saved Successfully")

Reports Saved Successfully


In [14]:
performance

,portfolio_rank,company_id,recommendation,allocation_percent,composite_score,quality_score_100,momentum_score_100,normalized_risk_score,risk_category,trend_classification,initial_price,close_price,return_percent,performance
0,1,BEL,Strong Buy,32.39,73.189325,100.00,85.71,6.67,Low Risk,Strong Improvement,1720.80,1998.73,16.15,Good
1,2,HAL,Buy,25.76,58.206714,79.45,42.86,13.33,Low Risk,Declining,4440.64,7687.57,73.12,Excellent
2,3,INDIGO,Hold,16.61,37.526171,51.09,57.14,6.67,Low Risk,Improving,1579.29,5956.44,277.16,Excellent
3,4,ABB,Hold,14.06,31.765038,43.19,71.43,13.33,Low Risk,Improving,1964.30,4913.07,150.12,Excellent
4,5,TRENT,Watch,11.19,25.274906,34.29,100.00,6.67,Low Risk,Strong Improvement,598.61,835.01,39.49,Excellent


In [15]:
summary

,Metric,Value
0,Companies,5.000
1,Average Return,111.208
2,Maximum Return,277.160
3,Minimum Return,16.150
